# Ujala | AI-01 and AI-02: Emergency recommendation demo
Rules-based demo logic. **Synthetic data only. Not clinical advice, not a diagnosis.**

**How to run:** open this notebook from inside `ml/recommendation/`, then *Run All*.
Needs `pandas` (`pip install pandas`). The real logic lives in `rules.py` and `recommender.py`, so it also works outside this notebook.

In [1]:
import json
import pandas as pd
from recommender import recommend, TABLE
from rules import load_rules, REQUIRED_TYPES, VALID_AMBULANCE
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200)

## 1. The rules table (AI-01)

In [2]:
rules_df = pd.read_csv("data/ujala_emergency_rules_synthetic.csv")
rules_df[["emergency_type", "ambulance_category", "hospital_category", "input_status"]]

,emergency_type,ambulance_category,hospital_category,input_status
0,cardiac,ALS,cardiac_capable_emergency_hospital,valid
1,accident_trauma,ALS,trauma_capable_emergency_hospital,valid
2,stroke,ALS,stroke_capable_emergency_hospital,valid
3,maternity,BLS,maternity_emergency_hospital,valid
4,breathing,ALS,emergency_hospital_with_respiratory_support,valid
5,burns,ALS,burns_capable_emergency_hospital,valid
6,general,BLS,general_emergency_hospital,valid
7,unknown,REQUIRES_TRIAGE,nearest_verified_emergency_service,unknown_type
8,NaN,REQUIRES_TRIAGE,nearest_verified_emergency_service,incomplete
9,cardiac,ALS,cardiac_capable_emergency_hospital,valid_duplicate_test


### Data quality checks: missing fields, duplicates, invalid categories

In [3]:
print("Rows:", len(rules_df))
print("Missing values per column:"); print(rules_df.isna().sum()[rules_df.isna().sum() > 0])
key = ["emergency_type", "ambulance_category", "hospital_category"]
print("\nDuplicate mappings:"); display(rules_df[rules_df.duplicated(key, keep=False)][key + ["input_status"]])
conflict = rules_df.dropna(subset=["emergency_type"]).groupby("emergency_type")[["ambulance_category"]].nunique()
print("Types with conflicting ambulance categories:", list(conflict[conflict.ambulance_category > 1].index))
print("Invalid ambulance categories:", set(rules_df.ambulance_category) - VALID_AMBULANCE)
print("Required types missing:", REQUIRED_TYPES - set(rules_df.emergency_type.dropna()))
print("All rows synthetic:", bool(rules_df.is_synthetic.all()))

Rows: 12
Missing values per column:
emergency_type    1
dtype: int64

Duplicate mappings:


,emergency_type,ambulance_category,hospital_category,input_status
0,cardiac,ALS,cardiac_capable_emergency_hospital,valid
9,cardiac,ALS,cardiac_capable_emergency_hospital,valid_duplicate_test


Types with conflicting ambulance categories: ['maternity']
Invalid ambulance categories: set()
Required types missing: set()
All rows synthetic: True


### How the loader resolves those issues
The exact duplicate `cardiac` row is ignored. `maternity` defaults to BLS; the specialist variant is only used when `specialist=True`. Unknown and blank types fall back to `REQUIRES_TRIAGE`.

In [4]:
for t, r in TABLE["rules"].items():
    print(f"{t:25} -> {r['ambulance']:10} | {r['hospital']}")
print("\nvariants:", {k: [v['ambulance'] for v in vs] for k, vs in TABLE['variants'].items()})

cardiac                   -> ALS        | cardiac_capable_emergency_hospital
accident_trauma           -> ALS        | trauma_capable_emergency_hospital
stroke                    -> ALS        | stroke_capable_emergency_hospital
maternity                 -> BLS        | maternity_emergency_hospital
breathing                 -> ALS        | emergency_hospital_with_respiratory_support
burns                     -> ALS        | burns_capable_emergency_hospital
general                   -> BLS        | general_emergency_hospital
critical_care_transport   -> ICU        | hospital_with_critical_care

variants: {'maternity': ['NEONATAL_OR_MATERNITY_SPECIALIST']}


## 2. Run the 12 synthetic cases (AI-02)

In [5]:
cases = pd.read_csv("data/ujala_synthetic_recommendation_cases.csv")
val = lambda x: None if pd.isna(x) else x

def to_request(r):
    amb = [] if val(r.ambulance_id) is None else [{"id": r.ambulance_id, "category": r.ambulance_category, "available": bool(r.ambulance_available)}]
    hos = [] if val(r.hospital_id) is None else [{"id": r.hospital_id, "category": r.hospital_category, "distanceKm": r.distance_km, "freeBeds": int(r.free_beds), "pmjay": bool(r.pmjay_match)}]
    return dict(emergency_type=val(r.emergency_type), lat=val(r.patient_lat), lng=val(r.patient_lng), ambulances=amb, hospitals=hos)

rows = []
for r in cases.itertuples():
    res = recommend(**to_request(r), specialist=(r.case_id == "CASE012"))
    rows.append({"case": r.case_id, "scenario": r.scenario_label, "ambulanceType": res["ambulanceType"],
                 "hospitalCategory": res["hospitalCategory"], "topHospitals": [h["id"] for h in res["topHospitals"]]})
pd.DataFrame(rows)

,case,scenario,ambulanceType,hospitalCategory,topHospitals
0,CASE001,normal,ALS,cardiac_capable_emergency_hospital,[HOSP001]
1,CASE002,normal,ALS,trauma_capable_emergency_hospital,[HOSP002]
2,CASE003,normal,ALS,stroke_capable_emergency_hospital,[HOSP003]
3,CASE004,normal,BLS,maternity_emergency_hospital,[HOSP004]
4,CASE005,normal,ALS,emergency_hospital_with_respiratory_support,[HOSP005]
5,CASE006,normal,ALS,burns_capable_emergency_hospital,[HOSP006]
6,CASE007,normal,BLS,general_emergency_hospital,[HOSP007]
7,CASE008,unavailable_ambulance_and_no_beds,ALS,cardiac_capable_emergency_hospital,[]
8,CASE009,unknown_emergency,REQUIRES_TRIAGE,nearest_verified_emergency_service,[HOSP009]
9,CASE010,incomplete_input,REQUIRES_TRIAGE,nearest_verified_emergency_service,[]


## 3. Ranking with a bigger candidate pool

In [6]:
amb = json.load(open("data/ambulances_pool_synthetic.json"))
hos = json.load(open("data/hospitals_pool_synthetic.json"))
PATIENT = (22.7196, 75.8577)

res = recommend("cardiac", *PATIENT, amb, hos)
print(res["ambulanceType"], "|", res["hospitalCategory"]); print(res["reason"])
pd.DataFrame(res["topHospitals"])

ALS | cardiac_capable_emergency_hospital
Demo mapping: cardiac symptoms may require advanced prehospital support and a hospital with cardiac emergency capability. 3 compatible available ambulance(s). (Demo logic, not clinical advice.)


,id,name,distanceKm,freeBeds,pmjay,score
0,HOSPP001,Demo Heart Institute,4.082700,6,True,0.5653
1,HOSPP002,Demo Cardiac Clinic,4.695553,1,False,0.0500


In [7]:
# every required emergency type against the same pool
out = []
for t in sorted(REQUIRED_TYPES):
    r = recommend(t, *PATIENT, amb, hos)
    out.append({"type": t, "ambulance": r["ambulanceType"], "top hospital": r["topHospitals"][0]["name"] if r["topHospitals"] else None})
pd.DataFrame(out)

,type,ambulance,top hospital
0,accident_trauma,ALS,Demo Trauma Centre
1,breathing,ALS,Demo Chest Care
2,burns,ALS,Demo Burns Unit
3,cardiac,ALS,Demo Heart Institute
4,general,BLS,Demo General North
5,maternity,BLS,Demo Mother Hospital
6,stroke,ALS,Demo Stroke Unit


### Weights come from `config.json`: changing them changes the ranking

In [8]:
for w in [{"w_distance": 1, "w_beds": 0, "w_pmjay": 0}, {"w_distance": 0, "w_beds": 1, "w_pmjay": 0}]:
    r = recommend("general", *PATIENT, amb, hos, config=w)
    print(w, "->", [h["name"] for h in r["topHospitals"]])

{'w_distance': 1, 'w_beds': 0, 'w_pmjay': 0} -> ['Demo Community Centre', 'Demo General North']
{'w_distance': 0, 'w_beds': 1, 'w_pmjay': 0} -> ['Demo General North', 'Demo Community Centre']


## 4. Edge cases: unknown, missing, empty lists

In [9]:
for label, args in {"unknown type": ("banana", *PATIENT, amb, hos),
                    "missing type": (None, *PATIENT, amb, hos),
                    "empty lists": ("cardiac", *PATIENT, [], []),
                    "None lists": ("cardiac", *PATIENT, None, None)}.items():
    r = recommend(*args)
    print(f"{label:14} -> {r['ambulanceType']:16} hospitals={len(r['topHospitals'])} | {r['reason'][:90]}...")

unknown type   -> REQUIRES_TRIAGE  hospitals=3 | Safe fallback: emergency type is unknown, so do not infer a specific ambulance level or sp...
missing type   -> REQUIRES_TRIAGE  hospitals=3 | Safe fallback: emergency type is missing; request clarification while directing the user t...
empty lists    -> ALS              hospitals=0 | Demo mapping: cardiac symptoms may require advanced prehospital support and a hospital wit...
None lists     -> ALS              hospitals=0 | Demo mapping: cardiac symptoms may require advanced prehospital support and a hospital wit...


## Limitations
- Demo rules only; not clinical advice. Real triage depends on the patient's condition and local protocols.
- Field names (`category`, `available`, `freeBeds`, `pmjay`, `distanceKm`) are placeholders until agreed with Naga Siddeswar.
- Exact-match ambulance compatibility (no ICU-covers-ALS upgrade).
- Hospitals with 0 free beds are excluded (configurable).